# 01 · Ympäristö ja testit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markomanninen/tgd-geometry-lab/blob/main/notebooks/01_ymparisto_ja_testit.ipynb)

Tämä kirja on rakennettu Lanyonin Colab-esimerkin (*Run LanyonScripts in Colab*) mallin mukaan. Siinä ovat
väliaikainen Ubuntu-ajoympäristö, GNU Maxima, Prodanovin `clifford`/`cliffordan` -paketit ja
skriptien ajo omissa Maxima-prosesseissaan.

Erona Lanyonin esimerkkiin on, että jokainen moduuli tarkistetaan **automaattisilla testeillä**
(tunnetut identiteetit kuten $d^2=0$, $d\omega=0$, $\mathrm{Ric}=3g$, Bianchi), jotta
laskentakirjaston virheet eivät sekoitu TGD:n fysikaalisiin oletuksiin (välitavoite M1).

In [1]:
# Asennus: Maxima, tämä repositorio ja Dimiter Prodanovin clifford-paketti.
# Colabissa ajo kestää noin minuutin. Paikallisesti asetetaan TGD_ROOT / CLIFFORD_PATH.
import os, sys, shutil
ROOT = os.environ.get("TGD_ROOT", "/content/tgd-geometry-lab")
CLI  = os.environ.get("CLIFFORD_PATH", "/content/clifford")
if not shutil.which("maxima"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y maxima > /dev/null
if not os.path.exists(ROOT):
    !git clone -q --depth 1 https://github.com/markomanninen/tgd-geometry-lab.git {ROOT}
if not os.path.exists(CLI):
    !git clone -q --depth 1 https://github.com/dprodanov/clifford.git {CLI}
sys.path.insert(0, ROOT + "/notebooks")
from tgdlab import Lab
lab = Lab(ROOT, CLI)
!maxima --version 2>/dev/null | tail -1

Maxima 5.50.0


## Testisarja

Ajaa kaikki `tests/`-hakemiston testit (metriikka, kaarevuus, muodot, $CP_2$, upotus, toiminta, Clifford).

In [2]:
ok = lab.tests()
print('KAIKKI TESTIT LÄPI' if ok else 'TESTEJÄ EPÄONNISTUI')

 pure Kahler term does not vanish 
  PASS  holomorphic in w: covariant E_5 = definition, c_K part non-zero 
  PASS  rotating geodesic: J_ux # 0 
  PASS  rotating geodesic: only j^v is non-zero 
  PASS  rotating geodesic: volume term vanishes 
  PASS  rotating geodesic: Kahler term vanishes 
--- tgd/noether: isometries of H and conserved currents --- 
  PASS  su(3): generators anti-Hermitian and traceless 
  PASS  CP_2: all 8 generators are Killing 
  PASS  CP_2: all 8 generators preserve J 
  PASS  control: a coordinate dilation is not Killing 
  PASS  su(3) closes: [K_A, K_B] = -K_[A,B] 
  PASS  moment map: i_K J = d phi for all 8 generators 
  PASS  H has 18 isometries here 
  PASS  H: all 18 preserve h_kl 
  PASS  H: all 18 preserve J_kl 
  PASS  D_a j^a = 2 K.Eq off shell, both sides non-zero: translation t 
  PASS  D_a j^a = 2 K.Eq off shell, both sides non-zero: boost t-x 
  PASS  D_a j^a = 2 K.Eq off shell, both sides non-zero: color lambda_1 
  PASS  D_a j^a = 2 K.Eq off shell,

## Clifford-algebra (Prodanov)

Sama paketti kuin LanyonScriptsissä. Tarkistetaan Clifford-relaatiot
$\gamma_a\gamma_b+\gamma_b\gamma_a = 2\eta_{ab}$ signatuurilla $(1,3)$.

In [3]:
print(lab.values("clifford(g, 1, 3)$",
    ["g[1] . g[1]", "g[2] . g[2]", "g[1] . g[2] + g[2] . g[1]", "g[3] . g[4] + g[4] . g[3]"],
    load_clifford=True))

['1', '-1', '0', '0']


## Vertailu LanyonScriptsin esimerkkeihin

Tämä vaihe on valinnainen. Siinä ajetaan Lanyonin kuusi virtausfysiikan esimerkkiä (Burgers, matala vesi, Euler, Maxwell, MHD)
samalla tavalla kuin alkuperäisessä kirjassa. Ne käyttävät samaa Maxima–Clifford-pohjaa,
jonka päälle TGD Geometry Lab rakentaa differentiaaligeometrian.

In [4]:
RUN_LANYON = os.environ.get("RUN_LANYON", "1") == "1"   # aseta 0 ohittaaksesi
if RUN_LANYON:
    import subprocess
    from pathlib import Path
    L = Path("/content/LanyonScripts")
    if not L.exists():
        !git clone -q --depth 1 https://github.com/lanyonai/LanyonScripts.git {L}
    for f in ["burgers.mac", "shallow-water.mac", "isoeuler.mac", "euler.mac", "maxwell.mac", "ideal-mhd.mac"]:
        code = (f'file_search_maxima: append(file_search_maxima, ["{CLI}/###.{{mac,wxm}}", "{CLI}/$$$.{{mac,wxm}}"])$ '
                f'load("{L}/eqnsys/{f}")$ print("LANYON_SCRIPT_FINISHED")$')
        p = subprocess.run(["maxima", "--very-quiet", f"--batch-string={code}"],
                           capture_output=True, text=True, timeout=300)
        print(f"{f:20s}", "OK" if "LANYON_SCRIPT_FINISHED" in p.stdout else "VIRHE")